# 02 – Framework test: Iris, Wine, Breast Cancer


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, time
from pathlib import Path

# Make `src` importable whether the notebook is started from notebooks/ or from the project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from src.datasets import load_dataset, dataset_overview
from src.experiments import ExperimentConfig, build_method, run_experiment, run_alpha2_grid_search, METHODS
from src.stats_tests import run_statistical_tests

pd.set_option("display.precision", 4)
DATASETS = ["iris", "wine", "breast_cancer"]
RESULTS_DIR = ROOT / "results"
print("Project root:", ROOT)

## 1. Dataset loader

In [ ]:
overview = dataset_overview()
overview[overview["key"].isin(DATASETS)]

In [ ]:
data = {}
for key in DATASETS:
    ds = load_dataset(key, strict=True)          # strict: fail if the file does not look as expected
    X = ds.preprocessed()                        # median imputation -> constant-column removal -> z-score
    data[key] = (ds, X)
    print(f"{ds.name:26s} raw {ds.X.shape}, missing={int(np.isnan(ds.X).sum()):3d} -> preprocessed {X.shape}, "
          f"k={ds.k}, classes={ds.class_names}, mean={X.mean():.2e}, std={X.std():.2f}")

## 2. One run of every method (seed 0, fixed k)

In [ ]:
cfg = ExperimentConfig()   # default hyperparameters from the project plan
rows = []
for key, (ds, X) in data.items():
    for method in METHODS:
        t0 = time.perf_counter()
        labels = build_method(method, ds.k, "fixed", seed=0, cfg=cfg).fit_predict(X)
        rows.append({"dataset": key, "method": method,
                     "ARI": adjusted_rand_score(ds.y, labels),
                     "NMI": normalized_mutual_info_score(ds.y, labels),
                     "clusters found": len(np.unique(labels)),
                     "time [s]": time.perf_counter() - t0})
single = pd.DataFrame(rows)
single.pivot(index="dataset", columns="method", values="ARI")[list(METHODS)]

## 3. Inside ACE and Fuzzy-ACE (Iris, variable k)

In [ ]:
ds, X = data["iris"]
for method in ["ACE", "Fuzzy-ACE"]:
    ace = build_method(method, ds.k, "variable", seed=0, cfg=cfg)
    theta = ace.stage1_transformation(X)
    sizes = np.bincount(ace.partition_ids_)
    theta_1, theta_2 = ace.stage2_consensus_merging(theta, X.shape[0])
    labels = ace.stage3_hard_clustering(theta_1, theta_2)
    print(f"--- {method}")
    print(f"Stage 1: Theta_1 is {theta.shape}; clusters per base partition = {sizes.tolist()}")
    print(f"         membership values: {'binary' if np.isin(theta, [0, 1]).all() else 'continuous in [0, 1]'}")
    print(f"Stage 2: {theta_1.shape[1]} candidate clusters kept, {theta_2.shape[1]} eliminated, "
          f"final alpha_1 = {ace.final_alpha_1_:.2f}")
    print(f"Stage 3: cluster sizes = {np.bincount(labels).tolist()}, "
          f"ARI = {adjusted_rand_score(ds.y, labels):.4f}")

## 4. Experiment runner

Both settings, all five methods, 10 runs each. Every run is appended to `results/notebook_test/raw_results.csv` immediately;
re-running this cell skips runs that are already stored (set `resume=False` to start over).

In [ ]:
exp_cfg = ExperimentConfig(name="notebook_test", datasets=DATASETS, n_runs=10, n_jobs=-1,
                           results_dir=str(RESULTS_DIR))
t0 = time.perf_counter()
raw = run_experiment(exp_cfg, resume=True, verbose=False)
print(f"{len(raw)} runs stored, {int((raw['status'] != 'ok').sum())} failed, {time.perf_counter() - t0:.0f}s")
raw.head()

In [ ]:
exp_dir = RESULTS_DIR / "notebook_test"
for setting in ["fixed", "variable"]:
    for metric in ["ari", "nmi"]:
        print(f"\n{metric.upper()} – {setting} k")
        display(pd.read_csv(exp_dir / "tables" / f"{metric}_{setting}.csv", index_col=0))

In [ ]:
# Spread of the 10 runs per method
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey="row")
for r, setting in enumerate(["fixed", "variable"]):
    for c, key in enumerate(DATASETS):
        sub = raw[(raw.setting == setting) & (raw.dataset == key)]
        axes[r, c].boxplot([sub[sub.method == m]["ari"] for m in METHODS], tick_labels=list(METHODS))
        axes[r, c].set_title(f"{key} – {setting} k")
        axes[r, c].grid(alpha=0.3)
    axes[r, 0].set_ylabel("ARI")
plt.tight_layout()

## 5. Statistical tests (Iman–Davenport, Nemenyi, CD diagrams)

In [ ]:
results = run_statistical_tests(str(exp_dir), alpha=0.1)
pd.read_csv(exp_dir / "statistics" / "tests_summary.csv")

## 6. α₂ grid search

Stages 1–2 are computed once per run, Stage 3 is repeated for α₂ ∈ {0.5, …, 0.9}.


In [ ]:
grid_cfg = ExperimentConfig(name="notebook_alpha2", datasets=DATASETS, settings=["fixed"],
                            methods=["ACE", "Fuzzy-ACE"], n_runs=10, n_jobs=-1, results_dir=str(RESULTS_DIR))
grid_raw = run_alpha2_grid_search(grid_cfg, alpha_2_values=[0.5, 0.6, 0.7, 0.8, 0.9], verbose=False)
grid_dir = RESULTS_DIR / "notebook_alpha2"
pd.read_csv(grid_dir / "stability.csv").query("metric == 'ari'")

## 7. Where the results are stored

In [ ]:
for d in [exp_dir, grid_dir]:
    print(d.relative_to(ROOT))
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print("   ", p.relative_to(d))